<a href="https://colab.research.google.com/github/Tarayme/Tarayme/blob/main/wanvideo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q \
  "diffusers==0.40.0" \
  "transformers==5.16.1" \
  "gradio==6.24.0" \
  "accelerate>=1.10.0" \
  "bitsandbytes>=0.47.0" \
  "sentencepiece>=0.2.0" \
  "ftfy>=6.3.0" \
  "imageio>=2.37.0" \
  "imageio-ffmpeg>=0.6.0" \
  "safetensors>=0.5.0"

print("✅ Kurulum tamamlandı.")
print("Eğer bu Colab oturumunda daha önce diffusers/gradio import ettiysen:")
print("Çalışma Zamanı > Oturumu yeniden başlat yap, sonra Hücre 2'den devam et.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00
✅ Kurulum tamamlandı.
Eğer bu Colab oturumunda daha önce diffusers/gradio import ettiysen:
Çalışma Zamanı > Oturumu yeniden başlat yap, sonra Hücre 2'den devam et.


In [ ]:
import gc
import os
import psutil
import torch
import diffusers
import transformers
import gradio as gr

from diffusers import WanPipeline, AutoencoderKLWan
from diffusers.quantizers import PipelineQuantizationConfig
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ GPU yok.\n"
        "Colab > Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU seç."
    )

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
RAM_GB = psutil.virtual_memory().total / 1024**3

print("GPU:", GPU_NAME)
print(f"VRAM: {VRAM_GB:.1f} GB")
print(f"Sistem RAM: {RAM_GB:.1f} GB")
print("BF16 native:", torch.cuda.is_bf16_supported())
print("Torch:", torch.__version__)
print("Diffusers:", diffusers.__version__)
print("Transformers:", transformers.__version__)
print("Gradio:", gr.__version__)

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

# T4 = FP16.
# L4/A100 gibi yeni GPU gelirse BF16 kullanabilir.
COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

print("Kullanılacak dtype:", COMPUTE_DTYPE)

# Dev UMT5 text encoder'ı 8-bit tut.
# Video transformer'ına dokunmuyoruz:
# görsel kaliteyi mümkün olduğunca korumak istiyoruz.
quant_config = PipelineQuantizationConfig(
    quant_backend="bitsandbytes_8bit",
    quant_kwargs={
        "load_in_8bit": True
    },
    components_to_quantize=["text_encoder"]
)

print("VAE yükleniyor...")

vae = AutoencoderKLWan.from_pretrained(
    MODEL_ID,
    subfolder="vae",
    dtype=torch.float32,
    low_cpu_mem_usage=True
)

print("Pipeline yükleniyor...")

pipe = WanPipeline.from_pretrained(
    MODEL_ID,
    vae=vae,
    quantization_config=quant_config,
    dtype=COMPUTE_DTYPE,
    low_cpu_mem_usage=True
)

# Diffusers 0.40 Wan örneğine uygun scheduler.
pipe.scheduler = UniPCMultistepScheduler.from_config(
    pipe.scheduler.config,
    flow_shift=5.0
)

# VAE decode aşamasında VRAM düşürür.
try:
    pipe.vae.enable_tiling()
    print("✅ VAE tiling açık")
except Exception as e:
    print("VAE tiling desteklenmedi:", e)

try:
    pipe.vae.enable_slicing()
    print("✅ VAE slicing açık")
except Exception:
    pass

# Bir seferde yalnız aktif ana bileşen GPU'da dursun.
pipe.enable_model_cpu_offload()

gc.collect()
torch.cuda.empty_cache()

print()
print("✅ WAN 2.1 T2V 1.3B hazır.")

GPU: Tesla T4
VRAM: 14.6 GB
Sistem RAM: 12.7 GB
BF16 native: True
Torch: 2.11.0+cu128
Diffusers: 0.40.0
Transformers: 5.16.1
Gradio: 6.24.0
Kullanılacak dtype: torch.bfloat16
VAE yükleniyor...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B /  508MB            

vae/diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

Pipeline yükleniyor...


model_index.json:   0%|          | 0.00/400 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/debugpy/_vendored/pydevd/_pydevd_bundle/pydevd_safe_repr.py:128: FutureWarning: Accessing config attribute `__iter__` directly via 'AutoencoderKLWan' object attribute is deprecated. Please access '__iter__' over 'AutoencoderKLWan's config object instead, e.g. 'unet.config.__iter__'.
  if not hasattr(obj, "__iter__"):
/usr/local/lib/python3.13/dist-packages/debugpy/_vendored/pydevd/_pydevd_bundle/pydevd_safe_repr.py:128: FutureWarning: Accessing config attribute `__iter__` directly via 'AutoencoderKLWan' object attribute is deprecated. Please access '__iter__' over 'AutoencoderKLWan's config object instead, e.g. 'unet.config.__iter__'.
  if not hasattr(obj, "__iter__"):


In [ ]:
import gc
import os
import uuid
import random
import traceback
import torch

from diffusers.utils import export_to_video

OUTPUT_DIR = "/content/wan_videos"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NEGATIVE_DEFAULT = """
overexposed, underexposed, static image, blurry, motion smear,
low quality, worst quality, jpeg artifacts, compression artifacts,
deformed body, malformed anatomy, distorted face, distorted hands,
extra fingers, fused fingers, missing fingers, extra limbs,
duplicate person, duplicate body parts, warped body,
unnatural movement, broken joints, flickering, temporal inconsistency,
text, subtitles, watermark, logo
""".strip()

# Hepsi 32'nin katı ve 480p civarı toplam piksel alanında.
SIZES = {
    "16:9  •  832×480": (480, 832),
    "9:16  •  480×832": (832, 480),
    "1:1   •  640×640": (640, 640),
}

# Wan kuralı: 4*k + 1
FRAMES = {
    "49 frame  •  ~3.1 sn": 49,
    "65 frame  •  ~4.1 sn": 65,
    "81 frame  •  ~5.1 sn": 81,
}

QUALITY = {
    "Hızlı test": 28,
    "Dengeli": 40,
    "Yüksek kalite": 50,
}

def clean_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def generate_video(
    prompt,
    negative_prompt,
    aspect,
    frame_setting,
    quality,
    guidance,
    seed
):
    prompt = (prompt or "").strip()

    if not prompt:
        raise gr.Error("Prompt boş olamaz.")

    negative_prompt = (negative_prompt or "").strip()
    if not negative_prompt:
        negative_prompt = NEGATIVE_DEFAULT

    height, width = SIZES[aspect]
    num_frames = FRAMES[frame_setting]
    steps = QUALITY[quality]

    try:
        seed = int(seed)
    except Exception:
        seed = -1

    if seed < 0:
        seed = random.randint(0, 2_147_483_647)

    generator = torch.Generator(
        device="cpu"
    ).manual_seed(seed)

    clean_memory()

    try:
        with torch.inference_mode():
            result = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                height=height,
                width=width,
                num_frames=num_frames,
                num_inference_steps=steps,
                guidance_scale=float(guidance),
                generator=generator,
                max_sequence_length=512,
            ).frames[0]

        filename = (
            f"wan_"
            f"{width}x{height}_"
            f"{num_frames}f_"
            f"seed{seed}_"
            f"{uuid.uuid4().hex[:6]}.mp4"
        )

        path = os.path.join(
            OUTPUT_DIR,
            filename
        )

        # Wan'ın standart playback değeri.
        FPS = 16

        export_to_video(
            result,
            path,
            fps=FPS
        )

        duration = num_frames / FPS

        del result
        clean_memory()

        info = (
            f"✅ Tamamlandı\n"
            f"Seed: {seed}\n"
            f"Çözünürlük: {width}×{height}\n"
            f"Frame: {num_frames}\n"
            f"FPS: {FPS}\n"
            f"Süre: ~{duration:.2f} sn\n"
            f"Steps: {steps}\n"
            f"CFG: {float(guidance):.1f}"
        )

        return path, info

    except torch.cuda.OutOfMemoryError:
        clean_memory()

        raise gr.Error(
            "❌ GPU belleği yetmedi.\n\n"
            "Önce 49 frame + 16:9 + Hızlı Test dene.\n"
            "Colab oturumunda eski modeller kaldıysa Runtime Restart yap."
        )

    except RuntimeError as e:
        clean_memory()

        if "out of memory" in str(e).lower():
            raise gr.Error(
                "❌ CUDA / RAM yetersiz.\n"
                "49 frame ile tekrar dene."
            )

        raise gr.Error(
            "Runtime hatası:\n" + str(e)
        )

    except Exception as e:
        clean_memory()
        traceback.print_exc()

        raise gr.Error(
            "Üretim hatası:\n" + str(e)
        )

In [ ]:
import gradio as gr

with gr.Blocks(
    title="Wan Video Studio"
) as app:

    gr.Markdown("""
# 🎬 Wan Video Studio

**Wan 2.1 T2V 1.3B — Colab Edition**

Daha stabil sonuç için promptu mümkünse İngilizce yaz.

İyi prompt sırası:

**subject → action → environment → camera → lighting → style**
""")

    with gr.Row():

        with gr.Column(scale=2):

            prompt = gr.Textbox(
                label="Prompt",
                lines=8,
                placeholder=(
                    "A young man walks slowly through a rainy neon-lit street "
                    "at night, natural body movement, realistic proportions, "
                    "the camera smoothly tracks him from the front, subtle handheld "
                    "motion, cinematic lighting, wet asphalt reflections, shallow "
                    "depth of field, photorealistic, consistent face and clothing"
                )
            )

            negative_prompt = gr.Textbox(
                label="Negative Prompt",
                value=NEGATIVE_DEFAULT,
                lines=5
            )

        with gr.Column(scale=1):

            aspect = gr.Dropdown(
                choices=list(SIZES.keys()),
                value="16:9  •  832×480",
                label="Format"
            )

            frame_setting = gr.Dropdown(
                choices=list(FRAMES.keys()),
                value="49 frame  •  ~3.1 sn",
                label="Video uzunluğu"
            )

            quality = gr.Radio(
                choices=list(QUALITY.keys()),
                value="Dengeli",
                label="Kalite"
            )

            guidance = gr.Slider(
                minimum=3.5,
                maximum=7.0,
                value=5.0,
                step=0.1,
                label="CFG / Prompt bağlılığı"
            )

            seed = gr.Number(
                value=-1,
                precision=0,
                label="Seed (-1 = rastgele)"
            )

    generate_btn = gr.Button(
        "🎥 VİDEO ÜRET",
        variant="primary"
    )

    info = gr.Textbox(
        label="Üretim Bilgisi",
        interactive=False
    )

    video_output = gr.Video(
        label="Video"
    )

    generate_btn.click(
        fn=generate_video,
        inputs=[
            prompt,
            negative_prompt,
            aspect,
            frame_setting,
            quality,
            guidance,
            seed
        ],
        outputs=[
            video_output,
            info
        ],
        concurrency_limit=1
    )

app.queue(
    max_size=5,
    default_concurrency_limit=1
)

app.launch(
    share=True,
    debug=True,
    show_error=True
)

In [ ]:
Running on public URL: https://xxxxxxxx.gradio.live